# Authentication Mechanism 1

CEDA data - open/public http (no account needed) [easy: unauthenticated http]

Find some open/public data: [link](https://catalogue.ceda.ac.uk/?q=public+data&results_per_page=20&sort_by=relevance&objects_related_to_uuid=&permissions_option=any&geo_option=True&north_bound=&west_bound=&east_bound=&south_bound=&start_date=&end_date=&date_option=publication_date&start_date_pub=&end_date_pub=&accessCategory=public)


In [1]:
%load_ext autoreload
%autoreload 2

Download using wget


In [14]:
!wget -e robots=off --mirror --no-parent -r https://dap.ceda.ac.uk/badc/deposited2025/Methane_Clumped_Isotopologues_Data_Base_2025//

--2026-01-20 11:57:55--  https://dap.ceda.ac.uk/badc/deposited2025/Methane_Clumped_Isotopologues_Data_Base_2025//
Resolving dap.ceda.ac.uk (dap.ceda.ac.uk)... 130.246.128.69
Connecting to dap.ceda.ac.uk (dap.ceda.ac.uk)|130.246.128.69|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘dap.ceda.ac.uk/badc/deposited2025/Methane_Clumped_Isotopologues_Data_Base_2025/index.html’

dap.ceda.ac.uk/badc     [ <=>                ]     877  --.-KB/s    in 0s      

Last-modified header missing -- time-stamps turned off.
2026-01-20 11:57:55 (288 MB/s) - ‘dap.ceda.ac.uk/badc/deposited2025/Methane_Clumped_Isotopologues_Data_Base_2025/index.html’ saved [877]

--2026-01-20 11:57:55--  https://dap.ceda.ac.uk/badc/deposited2025/Methane_Clumped_Isotopologues_Data_Base_2025//00README_catalogue_and_licence.txt
Reusing existing connection to dap.ceda.ac.uk:443.
HTTP request sent, awaiting response... 200 OK
Length: 820 [text/plain]
Saving to: ‘dap.c

Bulk Download Options - download multiple files from a CEDA archive


In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from pathlib import Path

BASE_URL = "https://dap.ceda.ac.uk/badc/deposited2025/Methane_Clumped_Isotopologues_Data_Base_2025/"

session = requests.Session()
session.headers["User-Agent"] = "python-wget-equivalent"


def recursive_download(url, output_dir="download"):
    r = session.get(url)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    for link in soup.find_all("a"):
        href = link.get("href")
        if not href or href.startswith("?") or href.startswith("../"):
            continue

        full_url = urljoin(url, href)

        if href.endswith("/"):
            recursive_download(full_url, output_dir)
        else:
            parsed = urlparse(full_url)
            local_path = Path(output_dir) / parsed.path.lstrip("/")
            local_path.parent.mkdir(parents=True, exist_ok=True)

            print(f"Downloading {full_url}")
            with session.get(full_url, stream=True) as r2:
                r2.raise_for_status()
                with open(local_path, "wb") as f:
                    for chunk in r2.iter_content(8192):
                        f.write(chunk)


recursive_download(BASE_URL)
